# 01 · Universo

**Objetivo:** Definir os carros candidatos: FIPE de R$ 50–85 mil no mês mais recente, ano-modelo ≥ 2017 (sem zero km), só carros de passeio.

**Entradas:** histórico FIPE (fipeX, versão merged); `data/ref/utilitarios.csv`; `data/ref/modelo_base_excecoes.csv`; `data/ref/fabricantes.csv`

**Saídas:** `data/processed/universo.parquet` (um par código × ano-modelo por linha) e `data/processed/fipe_hist_universo.parquet` (histórico de preços só desses pares)

## Passos

- [x] Baixar o histórico e carregar só carros, só as colunas necessárias
- [x] Conferir a fonte: mês mais recente, zero km, chave única
- [x] Extrair atributos do nome (regex) e medir a cobertura
- [x] Definir `modelo_base` (primeiro token + exceções revisadas)
- [x] Padronizar o nome da fabricante (`fabricante`)
- [x] Aplicar `filtrar_universo` e revisar os excluídos
- [x] Checar duplicidade de (código FIPE, ano-modelo) e contagens
- [x] Salvar universo e histórico dos pares

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # permite importar src/ a partir de notebooks/

import pandas as pd
from src.config import carregar_premissas, RAIZ
from src.dados import baixar_fipe, carregar_fipe_carros
from src.features import atributos_com_cache, cobertura, definir_modelo_base
from src.universo import adicionar_fabricante, filtrar_universo

pd.set_option("display.max_rows", 200)
p = carregar_premissas()

## 1. Carga

Usamos o arquivo **merged** do fipeX (nome atual do modelo em todo o histórico). O split `train` do
Hugging Face soma a versão original e a merged, então cada preço aparece duas vezes: por isso não
usamos `load_dataset`. O filtro `tipo_veiculo == carro` é aplicado já na leitura.

In [ ]:
baixar_fipe(p)                 # só baixa se o arquivo ainda não estiver em data/raw
df = carregar_fipe_carros(p)
print(f"{len(df):,} linhas de carro | memória: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.info()

## 2. Conferências da fonte

- O mês mais recente precisa estar completo: comparamos o número de linhas com os meses anteriores.
- Zero km: no fipeX vem como `zero_km = True` com `ano_modelo` nulo (não como ano 32000).
- Chave: na FIPE o "ano-modelo" é ano **+ combustível**. No histórico antigo existe o mesmo
  código e ano com dois combustíveis; por isso a chave completa inclui `sigla_combustivel`.

In [ ]:
print("Linhas nos últimos meses de referência:")
print(df["mesano_referencia"].value_counts().sort_index().tail(4).to_string())

print("\nzero_km × ano_modelo nulo:")
print(pd.crosstab(df["zero_km"], df["ano_modelo"].isna(), colnames=["ano_modelo nulo"]))

chave = ["codigo_fipe", "ano_modelo", "mesano_referencia"]
print("\nRepetições da chave (código, ano, mês):     ", df.duplicated(chave).sum())
print("Repetições incluindo o combustível na chave:", df.duplicated(chave + ["sigla_combustivel"]).sum())

## 3. Atributos do nome (regex)

A extração roda só sobre os **nomes únicos** do mês mais recente (alguns milhares) e é juntada de
volta por `nome_modelo`. O cache em parquet é refeito sozinho se algum padrão mudar.

Atributo não encontrado fica **nulo**: por exemplo, a FIPE muitas vezes omite "Mec." em carros
manuais, então `transmissao` nula não quer dizer "não tem câmbio". Imputação, se necessária, é na etapa 02.

In [ ]:
ultimo = df[df["mesano_referencia"] == df["mesano_referencia"].max()].copy()
ultimo["nome_modelo"] = ultimo["nome_modelo"].astype(str)

atributos = atributos_com_cache(ultimo["nome_modelo"], RAIZ / p["dados"]["cache_regex"])
ultimo = ultimo.merge(atributos, on="nome_modelo", how="left")

u = p["universo"]
na_faixa = (~ultimo["zero_km"] & (ultimo["ano_modelo"] >= u["ano_modelo_min"])
            & ultimo["valor"].between(u["fipe_min"], u["fipe_max"])).fillna(False)
nomes_faixa = ultimo.loc[na_faixa, "nome_modelo"].unique()
print(f"{len(atributos):,} nomes no mês mais recente; {len(nomes_faixa):,} na faixa de preço/ano")
cobertura(atributos).join(cobertura(atributos[atributos["nome_modelo"].isin(nomes_faixa)]),
                          lsuffix="_mes", rsuffix="_faixa")

In [ ]:
# Amostra para conferência visual da extração
cols = ["nome_modelo", "motorizacao", "valvulas", "transmissao", "turbo", "combustivel_token",
        "carroceria_explicita", "cabine", "nome_modelo_limpo"]
atributos[atributos["nome_modelo"].isin(nomes_faixa)][cols].sample(15, random_state=0)

## 4. `modelo_base`

Regra: primeiro token do nome. Exceções em `data/ref/modelo_base_excecoes.csv` para modelos de duas
palavras ou nomes enganosos. Exemplo: "DUSTER OROCH" é a picape Oroch, não o Duster. A lista de
exceções saiu da revisão dos primeiros tokens na faixa de preço.

In [ ]:
excecoes = pd.read_csv(RAIZ / "data" / "ref" / "modelo_base_excecoes.csv")
ultimo["modelo_base"] = definir_modelo_base(ultimo["nome_modelo"], excecoes)
excecoes

## 4b. Fabricante

A FIPE escreve a mesma fabricante de formas diferentes (ex.: o TIGGO aparece em "Caoa Chery" e em
"Caoa Chery/Chery"). A tabela `data/ref/fabricantes.csv` liga cada `nome_marca` a um nome padronizado.
Se aparecer uma marca nova sem correspondência, a função para com erro em vez de deixar nulo.

In [ ]:
fabricantes = pd.read_csv(RAIZ / "data" / "ref" / "fabricantes.csv")
ultimo = adicionar_fabricante(ultimo, fabricantes)

# Só as marcas que mudam de nome, entre as presentes no mês mais recente
(ultimo[["nome_marca", "fabricante"]].astype(str).drop_duplicates()
 .query("nome_marca != fabricante").sort_values("fabricante").reset_index(drop=True))

## 5. Filtro do universo

`filtrar_universo` aplica a faixa (mês mais recente, usado, ano ≥ mínimo, FIPE na faixa) e marca o
motivo de exclusão de cada linha. Quando mais de um motivo vale, fica o primeiro desta ordem:
PCD → modelo utilitário → cabine → carroceria utilitária → token de carga no nome.

In [ ]:
utilitarios = pd.read_csv(RAIZ / "data" / "ref" / "utilitarios.csv")["modelo_base"].tolist()
universo, excluidos = filtrar_universo(ultimo, p, utilitarios)
print(f"Universo: {len(universo):,} pares | excluídos: {len(excluidos):,}")
excluidos["motivo_exclusao"].value_counts()

In [ ]:
# Revisão: quais modelos foram excluídos e por quê
excluidos.groupby(["motivo_exclusao", "modelo_base"]).size().rename("pares").to_frame()

In [ ]:
# Revisão: excluídos por token/cabine/carroceria (casos mais sujeitos a falso positivo)
excluidos.loc[excluidos["motivo_exclusao"] != "modelo utilitário (utilitarios.csv)",
              ["nome_marca", "nome_modelo", "motivo_exclusao"]]

## 6. Conferências do universo

In [ ]:
print("Pares (código, ano-modelo) repetidos:", universo.duplicated(["codigo_fipe", "ano_modelo"]).sum())
print("Modelos-base distintos:", universo["modelo_base"].nunique())
universo["valor"].describe().round(0)

In [ ]:
universo.groupby("fabricante").agg(
    pares=("codigo_fipe", "size"), modelos=("modelo_base", "nunique")
).sort_values("pares", ascending=False)

In [ ]:
universo["ano_modelo"].value_counts().sort_index()

In [ ]:
universo["modelo_base"].value_counts()

## 7. Salvar

- `universo.parquet`: um par (código, ano-modelo) por linha, com preço do mês mais recente e atributos.
- `fipe_hist_universo.parquet`: todo o histórico de preço só desses pares, para as etapas 02 e 03
  não precisarem recarregar os ~6 milhões de linhas.

In [ ]:
dir_proc = RAIZ / p["dados"]["dir_processed"]
dir_proc.mkdir(parents=True, exist_ok=True)
universo.to_parquet(dir_proc / "universo.parquet", index=False)

# Histórico ligado por (código, ano-modelo, combustível): a chave completa da FIPE.
chave_hist = ["codigo_fipe", "ano_modelo", "sigla_combustivel"]
pares = universo[chave_hist].astype({"codigo_fipe": str, "sigla_combustivel": str})
hist = df.astype({"codigo_fipe": str, "sigla_combustivel": str}).merge(pares, on=chave_hist)
hist.to_parquet(dir_proc / "fipe_hist_universo.parquet", index=False)

print(f"universo.parquet: {len(universo):,} linhas")
print(f"fipe_hist_universo.parquet: {len(hist):,} linhas, "
      f"de {hist['mesano_referencia'].min()} a {hist['mesano_referencia'].max()}")

## Decisões e observações

- **Fonte:** arquivo `fipex-prices-latest-merged.parquet`. O split `train` do Hugging Face soma as
  versões original e merged (9.631.829 linhas cada) e duplicava todos os preços no notebook antigo.
- **Zero km:** no fipeX vem como `zero_km = True` e `ano_modelo` nulo; excluímos por essa coluna
  (e também por ano 32000, por segurança).
- **Chave:** na FIPE o ano-modelo é ano + combustível. No universo, (código, ano-modelo) já é único;
  o histórico é ligado ao universo por (código, ano-modelo, combustível).
- **Mês de referência:** o mais recente do arquivo, com contagem de linhas em linha com os meses anteriores.
- **Regex:** padrões em `src/features.py`; fronteiras que aceitam tokens com ponto ("TIP.", "SED.");
  atributo ausente = nulo. Cobertura na faixa: motorização ~98%, combustível ~89%, válvulas ~85%,
  transmissão ~77% (manuais muitas vezes sem "Mec.").
- **modelo_base:** primeiro token + 10 exceções revisadas (`modelo_base_excecoes.csv`).
- **fabricante:** nome padronizado da marca (`fabricantes.csv`, todas as marcas do mês mais recente).
  Junta, por exemplo, "Caoa Chery" e "Caoa Chery/Chery" em Chery, e "GM - Chevrolet" vira Chevrolet.
  `nome_marca` original é mantida.
- **Exclusões novas em `utilitarios.csv`:** AMAROK, BERLINGO, PARTNER, V21, V22, V25, MINITRUCK,
  DELIVERY, E.CO e BUGGY (buggies não são carro de uso diário). **A revisar pelo autor.**
- **PCD:** regra mantida; não há versões PCD na faixa no mês atual.
- **Em aberto:** ONIX junta hatch e sedã (Onix Plus) no mesmo `modelo_base`; a carroceria fica em
  `carroceria_explicita`. Separar ou não fica para a etapa 02.